In [ ]:
# Calculate Average Price Feature from one_hot_mama.csv
import pandas as pd
import numpy as np

# Read the year-specific data
#one_hot_data = pd.read_csv('data/one_hot_mama.csv')

# # Find all price columns (YEAR_price_usd_tonne)
# price_cols = [col for col in one_hot_data.columns if col.endswith('_price_usd_tonne')]
# print(f"Found {len(price_cols)} price columns: {sorted(price_cols)}")

# # Calculate average price: sum of non-zero values divided by count of non-zero entries
# def calculate_avg_price(row):
#     """Calculate average price from year-specific price columns, excluding zeros and NaNs"""
#     prices = row[price_cols].replace(0, np.nan)  # Treat 0 as missing
#     non_zero_prices = prices.dropna()
    
#     if len(non_zero_prices) == 0:
#         return np.nan
#     else:
#         return non_zero_prices.sum() / len(non_zero_prices)

# # Apply to each row
# avg_price_feature = one_hot_data.apply(calculate_avg_price, axis=1)

# # Read the average_data file and add the new feature
# average_data = pd.read_csv('data/average_data.csv')
# average_data['avg_price_usd'] = avg_price_feature

# # Save the updated file
# average_data.to_csv('data/average_data.csv', index=False)

# print(f"\nAdded 'avg_price_usd' feature to average_data.csv")
# print(f"New feature statistics:")
# print(average_data['avg_price_usd'].describe())
# print(f"\nNull values: {average_data['avg_price_usd'].isna().sum()}")


Found 21 price columns: ['2000_price_usd_tonne', '2001_price_usd_tonne', '2002_price_usd_tonne', '2003_price_usd_tonne', '2004_price_usd_tonne', '2005_price_usd_tonne', '2006_price_usd_tonne', '2007_price_usd_tonne', '2008_price_usd_tonne', '2009_price_usd_tonne', '2010_price_usd_tonne', '2011_price_usd_tonne', '2012_price_usd_tonne', '2013_price_usd_tonne', '2014_price_usd_tonne', '2015_price_usd_tonne', '2016_price_usd_tonne', '2017_price_usd_tonne', '2018_price_usd_tonne', '2019_price_usd_tonne', '2020_price_usd_tonne']

Added 'avg_price_usd' feature to average_data.csv
New feature statistics:
count      352.000000
mean      1791.852012
std       2915.813909
min         13.687000
25%         76.460536
50%        101.400000
75%       1894.250571
max      19845.493636
Name: avg_price_usd, dtype: float64

Null values: 22


# Regressor

In [8]:
import pandas as pd
import re
from itertools import combinations

In [ ]:
#average

#year specific 
# preferred_order =[   
#     'commodities_value_tonnes',
#     'coal_mined_value_tonnes',
#     'commodities_grade_ppm',
#     'commodities_recovery_rate',
#     'minerals_ore_mined_value_tonnes',
#     'price_usd_tonnes',
#     'reserves_mineral_value_tonnes',
#     'reserves_commodity_value_tonnes',
#     'reserves_grade_ppm',
#     'primary_commodity',
#     'mining_salary',
#     'still_operating',
#     'country',
# ]


# feature_group_names = sorted(preferred_order)
# print(f"Preferred order has {len(preferred_order)} groups")

Preferred order has 13 groups


In [ ]:
mother_file = pd.read_csv('data/average_data.csv') #average


{'total_reserves_minerals': ['total_reserves_minerals'], 'total_reserves_commodities': ['total_reserves_commodities'], 'avg_reserves_grade_ppm': ['avg_reserves_grade_ppm'], 'avg_commodity_recovery_rate': ['avg_commodity_recovery_rate'], 'avg_commodities_grade_ppm': ['avg_commodities_grade_ppm'], 'total_coal': ['total_coal'], 'total_commodities': ['total_commodities'], 'total_minerals': ['total_minerals'], 'mining_salary': ['mining_salary'], 'still_operating': ['still_operating'], 'country': ['country_Argentina', 'country_Australia', 'country_Bolivia', 'country_Bosnia and Herzegovina', 'country_Brazil', 'country_Canada', 'country_Chile', 'country_China', 'country_Colombia', "country_Cote d'Ivoire", 'country_Cuba', 'country_DR Congo', 'country_Dominican Republic', 'country_Finland', 'country_Ghana', 'country_Guatemala', 'country_Guinea', 'country_India', 'country_Indonesia', 'country_Jamaica', 'country_Kazakhstan', 'country_Kyrgyzstan', 'country_Liberia', 'country_Madagascar', 'country_M

In [20]:
# Group average_data features and exclude target/data identifiers
import re

average_data = pd.read_csv('data/average_data.csv')

YEAR_PREFIX_RE = re.compile(r'^(\d{4})_(.+)$')
ignore_cols = {'facility_id', 'LOM', 'LOM_in_Buckets', 'LOM_Buckets', 'LOM_quintile'}
feature_groups = {}

for col in average_data.columns:
    if col in ignore_cols:
        continue

    if col.startswith('country_'):
        group_name = 'country'
    elif col.startswith('primary_commodity_'):
        group_name = 'primary_commodity'
    else:
        match = YEAR_PREFIX_RE.match(col)
        group_name = match.group(2) if match else col

    feature_groups.setdefault(group_name, []).append(col)

feature_group_names = sorted(feature_groups)
print(f"Created {len(feature_group_names)} feature groups from average_data.csv")
print('Example groups:')
for group_name in ('country', 'primary_commodity'):
    if group_name in feature_groups:
        print(f"  {group_name}: {len(feature_groups[group_name])} columns")

print(f"\nAll feature groups: {feature_group_names}")


Created 14 feature groups from average_data.csv
Example groups:
  country: 47 columns
  primary_commodity: 11 columns

All feature groups: ['avg_commodities_grade_ppm', 'avg_commodity_recovery_rate', 'avg_price_usd', 'avg_reserves_grade_ppm', 'commodity_var', 'country', 'mining_salary', 'primary_commodity', 'still_operating', 'total_coal', 'total_commodities', 'total_minerals', 'total_reserves_commodities', 'total_reserves_minerals']


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import heapq
from itertools import count
import math

# Load the average_data file and target variable
average_data = pd.read_csv('data/average_data.csv')
target = 'LOM_in_Buckets'

search_groups = sorted(feature_groups)
max_results = 20
best_results = []
tie_breaker = count()

total_combos = sum(math.comb(len(search_groups), r) for r in range(1, len(search_groups) + 1))

for r in range(1, len(search_groups) + 1):
    for combo in tqdm(combinations(search_groups, r), total=math.comb(len(search_groups), r), desc=f'Group size {r}', leave=False):
        selected_cols = [col for group in combo for col in feature_groups[group]]
        X = average_data[selected_cols].select_dtypes(include='number').copy()
        y = average_data[target].copy()

        # Replace infinite values and drop rows with missing values
        X = X.replace([np.inf, -np.inf], np.nan)
        valid_mask = X.notna().all(axis=1) & y.notna()
        X = X.loc[valid_mask]
        y = y.loc[valid_mask]

        if X.shape[0] < 10 or X.shape[1] == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        model = RandomForestRegressor(random_state=42, n_estimators=100)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        score = r2_score(y_test, y_pred)

        record = {
            'feature_groups': combo,
            'num_groups': len(combo),
            'num_columns': len(selected_cols),
            'num_rows': X.shape[0],
            'r2_test': score,
        }

        heap_item = (score, next(tie_breaker), record)
        if len(best_results) < max_results:
            heapq.heappush(best_results, heap_item)
        else:
            heapq.heappushpop(best_results, heap_item)

best_sorted = [record for _, _, record in sorted(best_results, key=lambda x: x[0], reverse=True)]
combo_df = pd.DataFrame(best_sorted)
print(combo_df.to_string(index=False))

combo_df.to_csv('data/feature_group_regression_results.csv', index=False)
print('Saved top 20 feature-group combination regression results to data/feature_group_regression_results.csv')


KeyboardInterrupt: 

# Classifier

In [ ]:
mother_file = pd.read_csv('data/average_data.csv')
mother_file.to_csv('data/average_data.csv', index=False)
preferred_order =[   
    'commodities_value_tonnes',
    'coal_mined_value_tonnes',
    'commodities_grade_ppm',
    'commodities_recovery_rate',
    'minerals_ore_mined_value_tonnes'
    'avg_price_usd_tonnes',
    'reserves_mineral_value_tonnes',
    'reserves_commodity_value_tonnes',
    'reserves_grade_ppm',
    'primary_commodity',
    'mining_salary',
    'still_operating',
    'country',
]

remaining_groups = [
    g for g in feature_group_names
    if g not in preferred_order and g not in {'facility_id', 'LOM_Buckets', 'LOM_quintile'}
]

search_groups = [g for g in preferred_order if g in feature_group_names] + remaining_groups
print(f"Searching combinations over {len(search_groups)} feature groups")

best_results = []
max_results = 20

for r in range(1,  len(preferred_order)):
    for combo in combinations(search_groups, r):
        selected_cols = [col for group in combo for col in feature_groups[group]]
        X_train_combo = mother_file[selected_cols].select_dtypes(include='number').copy()
        X_test_combo = mother_file[selected_cols].select_dtypes(include='number').copy()

        X_train_combo = X_train_combo.replace([np.inf, -np.inf], np.nan)
        X_test_combo = X_test_combo.replace([np.inf, -np.inf], np.nan)


        # # CLASSIFIER
        # model_combo = RandomForestClassifier(
        #     n_estimators=100,
        #     max_depth=None,
        #     min_samples_leaf=2,
        #     class_weight="balanced",   # handles any class imbalance from uneven quintiles
        #     random_state=42,
        #     n_jobs=-1,
        # )
        # model_combo.fit(X_train_combo, y_train)
        # y_pred = model_combo.predict(X_test_combo)

        # accuracy = accuracy_score(y_test, y_pred)
        # balanced_acc = balanced_accuracy_score(y_test, y_pred)
        # report = classification_report(y_test, y_pred, output_dict=True)
        # f1_w = report['weighted avg']['f1-score']

        # record = {
        #     'feature_groups': combo,
        #     'num_columns': len(selected_cols),
        #     'accuracy': accuracy,
        #     'balanced_accuracy': balanced_acc,
        #     'f1_weighted': f1_w,
        #     'precision_weighted': report['weighted avg']['precision'],
        #     'recall_weighted': report['weighted avg']['recall'],
        # }

        # REGRESSOR
        model_combo = RandomForestRegressor(random_state=42, n_estimators=100)
        model_combo.fit(X_train_combo, y_train)
        score = r2_score(y_test, model_combo.predict(X_test_combo))

        record = {
            'feature_groups': combo,
            'num_columns': len(selected_cols),
            'r2_test': score,
        }

        if len(best_results) < max_results:
            heapq.heappush(best_results, (f1_w, record))
        else:
            heapq.heappushpop(best_results, (f1_w, record))

best_sorted = [record for score, record in sorted(best_results, key=lambda x: x[0], reverse=True)]
combo_df = pd.DataFrame(best_sorted)
print(combo_df.to_string(index=False))

# Save only the top results to keep the output file small.
combo_df.to_csv('data/feature_group_combo_results.csv', index=False)
print('Saved top 20 feature-group combo results to data/feature_group_combo_results.csv')